# Read the data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Financial_News_Sentiment_Prediction_using_DeepLearning/cleaned.csv")
df_test = pd.read_csv("/content/drive/MyDrive/Financial_News_Sentiment_Prediction_using_DeepLearning/cleaned_test.csv")


# Import necessary Libraries

In [ ]:
import nltk

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

import pandas as pd
import numpy as np
import torch
from torch import nn, optim
from sklearn.feature_extraction.text import TfidfVectorizer
from torch.utils.data import TensorDataset, DataLoader
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
train_df = df.copy()
test_df = df_test.copy()

# Check dataset sizes
print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

# Check label distribution
print("\nTraining label distribution:")
print(train_df["label"].value_counts())

print("\nTesting label distribution:")
print(test_df["label"].value_counts())


Training shape: (9543, 3)
Testing shape: (2388, 3)

Training label distribution:
label
2    6178
1    1923
0    1442
Name: count, dtype: int64

Testing label distribution:
label
2    1566
1     475
0     347
Name: count, dtype: int64


In [ ]:
#seperating features and target variable
# Training data
X_train = train_df["text"]
y_train = train_df["label"]

# Testing data
X_test = test_df["text"]
y_test = test_df["label"]

# Check shapes
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (9543,)
y_train: (9543,)
X_test: (2388,)
y_test: (2388,)


In [ ]:
print(type(X_train))
print(type(X_test))
print(type(y_train))
print(type(y_test))

<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>
<class 'pandas.core.series.Series'>


## data preparation

In [ ]:
import torch
import numpy as np

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

from sklearn.model_selection import train_test_split


# =========================================================
# 1. HANDLE MISSING TEXT VALUES
# =========================================================

# Keep X_train and X_test as Pandas Series

X_train = X_train.fillna("")
X_test = X_test.fillna("")


# =========================================================
# 2. BUILD VOCABULARY FROM TRAINING DATA ONLY
# =========================================================

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"


word_to_index = {
    PAD_TOKEN: 0,
    UNK_TOKEN: 1
}


# Build vocabulary using ONLY training text

for sentence in X_train:

    for word in sentence.split():

        if word not in word_to_index:

            word_to_index[word] = len(
                word_to_index
            )


vocab_size = len(
    word_to_index
)


print(
    "Vocabulary size:",
    vocab_size
)


# =========================================================
# 3. TEXT -> INTEGER SEQUENCE
# =========================================================

def text_to_sequence(text):

    return [
        word_to_index.get(
            word,
            word_to_index[UNK_TOKEN]
        )
        for word in text.split()
    ]


# Training sequences
train_sequences = X_train.apply(
    text_to_sequence
).tolist()


# Test sequences
# IMPORTANT:
# Uses the SAME vocabulary created from training data

test_sequences = X_test.apply(
    text_to_sequence
).tolist()


# =========================================================
# 4. FIND MAXIMUM SEQUENCE LENGTH
# =========================================================

# Calculate sequence lengths from TRAINING data only

train_lengths = [
    len(seq)
    for seq in train_sequences
]


max_length = max(
    train_lengths
)


print(
    "Maximum train length:",
    max(train_lengths)
)


print(
    "Selected max length:",
    max_length
)


# =========================================================
# 5. PAD / TRUNCATE SEQUENCES
# =========================================================

def pad_sequence(
    sequence,
    max_length
):

    # Truncate if sequence is too long
    sequence = sequence[
        :max_length
    ]

    # Pad if sequence is shorter, because equal-length sequences can be stacked into batches and passed to your neural network.
    sequence = sequence + [
        word_to_index[PAD_TOKEN]
    ] * (
        max_length - len(sequence)
    )

    return sequence


# =========================================================
# 6. CREATE FEATURE TENSORS
# =========================================================

# Should not overwrite X_train and X_test.

X_train_tensor = torch.tensor(
    [
        pad_sequence(
            seq,
            max_length
        )
        for seq in train_sequences
    ],
    dtype=torch.long
)


X_test_tensor = torch.tensor(
    [
        pad_sequence(
            seq,
            max_length
        )
        for seq in test_sequences
    ],
    dtype=torch.long
)


# =========================================================
# 7. CREATE LABEL TENSORS
# =========================================================

# Keep original y_train and y_test as Pandas Series

y_train_tensor = torch.tensor(
    y_train.to_numpy(),
    dtype=torch.long
)


y_test_tensor = torch.tensor(
    y_test.to_numpy(),
    dtype=torch.long
)


# =========================================================
# 8. CHECK ORIGINAL TENSOR SHAPES
# =========================================================

print(
    "\nOriginal Tensor Shapes:"
)


print(
    "X_train:",
    X_train_tensor.shape
)


print(
    "y_train:",
    y_train_tensor.shape
)


print(
    "X_test:",
    X_test_tensor.shape
)


print(
    "y_test:",
    y_test_tensor.shape
)


# =========================================================
# 9. CHECK ORIGINAL CLASS DISTRIBUTION
# =========================================================

print(
    "\nOriginal Training Distribution:"
)


print(
    torch.bincount(
        y_train_tensor
    )
)


print(
    "\nTest Distribution:"
)


print(
    torch.bincount(
        y_test_tensor
    )
)


# =========================================================
# 10. SPLIT ORIGINAL TRAINING DATA
#     INTO TRAINING + VALIDATION
# =========================================================

# Create indices for original training data

indices = np.arange(
    len(X_train_tensor)
)


# 80% training
# 20% validation
#
# stratify keeps approximately the same
# class proportions in both datasets

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    stratify=y_train_tensor.numpy()
)


# =========================================================
# 11. CREATE FINAL TRAINING DATA
# =========================================================

X_train_final = X_train_tensor[
    train_indices
]


y_train_final = y_train_tensor[
    train_indices
]


# =========================================================
# 12. CREATE VALIDATION DATA
# =========================================================

X_val = X_train_tensor[
    val_indices
]


y_val = y_train_tensor[
    val_indices
]


# =========================================================
# 13. CHECK TRAIN / VALIDATION / TEST SHAPES
# =========================================================

print(
    "\nAfter Train-Validation Split:"
)


print(
    "X_train_final:",
    X_train_final.shape
)


print(
    "y_train_final:",
    y_train_final.shape
)


print(
    "X_val:",
    X_val.shape
)


print(
    "y_val:",
    y_val.shape
)


print(
    "X_test:",
    X_test_tensor.shape
)


print(
    "y_test:",
    y_test_tensor.shape
)


# =========================================================
# 14. CHECK CLASS DISTRIBUTION AFTER SPLIT
# =========================================================

print(
    "\nFinal Training Distribution:"
)


print(
    torch.bincount(
        y_train_final
    )
)


print(
    "\nValidation Distribution:"
)


print(
    torch.bincount(
        y_val
    )
)


print(
    "\nTest Distribution:"
)


print(
    torch.bincount(
        y_test_tensor
    )
)


# =========================================================
# 15. CALCULATE CLASS WEIGHTS
# =========================================================

# IMPORTANT:
# Calculate weights ONLY from y_train_final.
#
# Do not use validation labels.
# Do not use test labels.

class_counts = torch.bincount(
    y_train_final
)


total_samples = len(
    y_train_final
)


num_classes = len(
    class_counts
)


class_weights = (
    total_samples
    /
    (
        num_classes
        *
        class_counts.float()
    )
)


print(
    "\nFinal Training Class Counts:"
)


print(
    class_counts
)


print(
    "\nClass Weights:"
)


print(
    class_weights
)


# =========================================================
# 16. CREATE DATASETS
# =========================================================

train_dataset = TensorDataset(
    X_train_final,
    y_train_final
)


val_dataset = TensorDataset(
    X_val,
    y_val
)


test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)


# =========================================================
# 17. SET BATCH SIZE
# =========================================================

batch_size = 64


# =========================================================
# 18. CREATE TRAINING DATALOADER
# =========================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)


# =========================================================
# 19. CREATE VALIDATION DATALOADER
# =========================================================

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)


# =========================================================
# 20. CREATE TEST DATALOADER
# =========================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)


# =========================================================
# 21. CHECK DATASET SIZES
# =========================================================

print(
    "\nDataset Sizes:"
)


print(
    "Training:",
    len(train_dataset)
)


print(
    "Validation:",
    len(val_dataset)
)


print(
    "Test:",
    len(test_dataset)
)


# =========================================================
# 22. VERIFY FIRST TRAINING BATCH
# =========================================================

X_batch, y_batch = next(
    iter(train_loader)
)


print(
    "\nFirst Training Batch:"
)


print(
    "X batch shape:",
    X_batch.shape
)


print(
    "y batch shape:",
    y_batch.shape
)


# =========================================================
# 23. VERIFY FIRST VALIDATION BATCH
# =========================================================

X_val_batch, y_val_batch = next(
    iter(val_loader)
)


print(
    "\nFirst Validation Batch:"
)


print(
    "X validation batch shape:",
    X_val_batch.shape
)


print(
    "y validation batch shape:",
    y_val_batch.shape
)


# =========================================================
# 24. VERIFY FIRST TEST BATCH
# =========================================================

X_test_batch, y_test_batch = next(
    iter(test_loader)
)


print(
    "\nFirst Test Batch:"
)


print(
    "X test batch shape:",
    X_test_batch.shape
)


print(
    "y test batch shape:",
    y_test_batch.shape
)

Vocabulary size: 31611
Maximum train length: 32
Selected max length: 32

Original Tensor Shapes:
X_train: torch.Size([9543, 32])
y_train: torch.Size([9543])
X_test: torch.Size([2388, 32])
y_test: torch.Size([2388])

Original Training Distribution:
tensor([1442, 1923, 6178])

Test Distribution:
tensor([ 347,  475, 1566])

After Train-Validation Split:
X_train_final: torch.Size([7634, 32])
y_train_final: torch.Size([7634])
X_val: torch.Size([1909, 32])
y_val: torch.Size([1909])
X_test: torch.Size([2388, 32])
y_test: torch.Size([2388])

Final Training Distribution:
tensor([1154, 1538, 4942])

Validation Distribution:
tensor([ 288,  385, 1236])

Test Distribution:
tensor([ 347,  475, 1566])

Final Training Class Counts:
tensor([1154, 1538, 4942])

Class Weights:
tensor([2.2051, 1.6545, 0.5149])

Dataset Sizes:
Training: 7634
Validation: 1909
Test: 2388

First Training Batch:
X batch shape: torch.Size([64, 32])
y batch shape: torch.Size([64])

First Validation Batch:
X validation batch shap

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", device)

Using device: cuda


# Training function

In [ ]:
import copy
import torch

from sklearn.metrics import f1_score


def train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    num_epochs=100,
    patience=5
):

    # =====================================================
    # EARLY STOPPING VARIABLES
    # =====================================================

    # Higher Macro F1 is better
    best_val_f1 = 0.0

    patience_counter = 0

    best_model_state = None


    # =====================================================
    # TRAINING LOOP
    # =====================================================

    for epoch in range(
        1,
        num_epochs + 1
    ):

        # =================================================
        # 1. TRAINING
        # =================================================

        model.train()

        running_loss = 0.0

        correct = 0
        total = 0


        for X_batch, y_batch in train_loader:

            # Move data to device
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)


            # ---------------------------------------------
            # Clear previous gradients
            # ---------------------------------------------

            optimizer.zero_grad()


            # ---------------------------------------------
            # Forward pass
            # ---------------------------------------------

            output = model(
                X_batch
            )


            # ---------------------------------------------
            # Calculate training loss
            # ---------------------------------------------

            loss = criterion(
                output,
                y_batch
            )


            # ---------------------------------------------
            # Backpropagation
            # ---------------------------------------------

            loss.backward()


            # ---------------------------------------------
            # Gradient clipping
            # ---------------------------------------------

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )


            # ---------------------------------------------
            # Update model weights
            # ---------------------------------------------

            optimizer.step()


            # ---------------------------------------------
            # Accumulate training loss
            # ---------------------------------------------

            running_loss += (
                loss.item()
                *
                X_batch.size(0)
            )


            # ---------------------------------------------
            # Training predictions
            # ---------------------------------------------

            predictions = torch.argmax(
                output,
                dim=1
            )


            correct += (
                predictions == y_batch
            ).sum().item()


            total += y_batch.size(0)


        # =================================================
        # 2. TRAINING METRICS
        # =================================================

        epoch_loss = (
            running_loss
            /
            total
        )


        epoch_accuracy = (
            correct
            /
            total
        )


        # =================================================
        # 3. VALIDATION
        # =================================================

        model.eval()


        val_running_loss = 0.0

        val_correct = 0
        val_total = 0


        # Store all validation predictions and labels
        # for calculating Macro F1

        val_predictions = []

        val_actual = []


        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                # Move validation data to device
                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)


                # -----------------------------------------
                # Forward pass
                # -----------------------------------------

                output = model(
                    X_batch
                )


                # -----------------------------------------
                # Validation loss
                # -----------------------------------------

                val_loss = criterion(
                    output,
                    y_batch
                )


                val_running_loss += (
                    val_loss.item()
                    *
                    X_batch.size(0)
                )


                # -----------------------------------------
                # Validation predictions
                # -----------------------------------------

                predictions = torch.argmax(
                    output,
                    dim=1
                )
                #dim=1 selects the class with the highest score for each tweet


                # -----------------------------------------
                # Validation accuracy
                # -----------------------------------------

                val_correct += (
                    predictions == y_batch
                ).sum().item()


                val_total += y_batch.size(0)


                # -----------------------------------------
                # Store predictions for Macro F1
                # -----------------------------------------

                val_predictions.extend(
                    predictions
                    .cpu()
                    .numpy()
                )


                val_actual.extend(
                    y_batch
                    .cpu()
                    .numpy()
                )


        # =================================================
        # 4. VALIDATION METRICS
        # =================================================

        epoch_val_loss = (
            val_running_loss
            /
            val_total
        )


        epoch_val_accuracy = (
            val_correct
            /
            val_total
        )


        # =================================================
        # 5. VALIDATION MACRO F1
        # =================================================

        epoch_val_f1 = f1_score(
            val_actual,
            val_predictions,
            average="macro",
            zero_division=0
        )


        # =================================================
        # 6. DISPLAY RESULTS
        # =================================================

        print(
            f"Epoch {epoch}/{num_epochs} "
            f"--> Train Loss: {epoch_loss:.4f} "
            f"--> Train Accuracy: {epoch_accuracy:.4f} "
            f"--> Val Loss: {epoch_val_loss:.4f} "
            f"--> Val Accuracy: {epoch_val_accuracy:.4f} "
            f"--> Val Macro F1: {epoch_val_f1:.4f}"
        )


        # =================================================
        # 7. EARLY STOPPING USING VALIDATION MACRO F1
        # =================================================

        # Higher F1 = better model

        if epoch_val_f1 > best_val_f1:

            # Save new best F1
            best_val_f1 = epoch_val_f1


            # Save model weights
            best_model_state = copy.deepcopy(
                model.state_dict()
            )


            # Reset patience because model improved
            patience_counter = 0


            print(
                f"Validation Macro F1 improved "
                f"to {best_val_f1:.4f}"
            )


        else:

            # No improvement
            patience_counter += 1


            print(
                f"No validation Macro F1 improvement "
                f"({patience_counter}/{patience})"
            )


        # =================================================
        # 8. CHECK EARLY STOPPING
        # =================================================

        if patience_counter >= patience:

            print(
                f"\nEarly stopping triggered "
                f"at epoch {epoch}"
            )


            print(
                f"Best Validation Macro F1: "
                f"{best_val_f1:.4f}"
            )


            break


    # =====================================================
    # 9. RESTORE BEST MODEL
    # =====================================================

    if best_model_state is not None:

        model.load_state_dict(
            best_model_state
        )


        print(
            "\nBest model weights restored."
        )


        print(
            f"Best Validation Macro F1: "
            f"{best_val_f1:.4f}"
        )


    # =====================================================
    # 10. RETURN BEST MODEL
    # =====================================================

    return model

## testing/evaluation stage

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import numpy as np
import torch


# =========================================================
# MODEL EVALUATION FUNCTION
# =========================================================

def evaluate_model(
    model,
    test_loader,
    device
):

    model.eval()

    all_predictions = []
    all_actual = []

    with torch.no_grad():

        for X_batch, y_batch in test_loader:

            # Move data to device
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            # Model prediction
            output = model(X_batch)

            # Get predicted class
            predictions = torch.argmax(
                output,
                dim=1
            )

            # Store results
            all_predictions.extend(
                predictions.cpu().numpy()
            )

            all_actual.extend(
                y_batch.cpu().numpy()
            )


    # =====================================================
    # METRICS
    # =====================================================

    accuracy = accuracy_score(
        all_actual,
        all_predictions
    )

    precision = precision_score(
        all_actual,
        all_predictions,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        all_actual,
        all_predictions,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        all_actual,
        all_predictions,
        average="macro",
        zero_division=0
    )


    print(
        f"Accuracy  : {accuracy:.4f}"
    )

    print(
        f"Precision : {precision:.4f}"
    )

    print(
        f"Recall    : {recall:.4f}"
    )

    print(
        f"F1 Score  : {f1:.4f}"
    )


    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================

    print("\nClassification Report:")

    print(
        classification_report(
            all_actual,
            all_predictions,
            target_names=[
                "Bearish",
                "Bullish",
                "Neutral"
            ],
            zero_division=0
        )
    )


    # =====================================================
    # CONFUSION MATRIX
    # =====================================================

    cm = confusion_matrix(
        all_actual,
        all_predictions
    )

    print("Confusion Matrix:")

    print(cm)


    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "confusion_matrix": cm,
        "actual": np.array(all_actual),
        "predictions": np.array(all_predictions)
    }

# Define models

In [ ]:
import torch.nn as nn


class RNNModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=100,
        hidden_dim=128,
        output_dim=3,
        dropout=0.3,
        bidirectional=False,
        padding_idx=0
    ):

        super().__init__()

        self.bidirectional = bidirectional


        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=padding_idx
        )


        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=bidirectional
        )


        self.dropout = nn.Dropout(
            dropout
        )


        direction = (
            2 if bidirectional
            else 1
        )


        self.fc = nn.Linear(
            hidden_dim * direction,
            output_dim
        )


    def forward(self, X):

        X = self.embedding(X)


        output, hidden = self.rnn(
            X
        )


        if self.bidirectional:

            hidden = torch.cat(
                (
                    hidden[-2],
                    hidden[-1]
                ),
                dim=1
            )

        else:

            hidden = hidden[-1]


        hidden = self.dropout(
            hidden
        )


        X = self.fc(
            hidden
        )


        return X

In [ ]:
class LSTMModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=100,
        hidden_dim=128,
        output_dim=3,
        dropout=0.3,
        bidirectional=False,
        padding_idx=0
    ):

        super().__init__()

        self.bidirectional = bidirectional


        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=padding_idx
        )


        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=bidirectional
        )


        self.dropout = nn.Dropout(
            dropout
        )


        direction = (
            2 if bidirectional
            else 1
        )


        self.fc = nn.Linear(
            hidden_dim * direction,
            output_dim
        )


    def forward(self, X):

        X = self.embedding(X)


        output, (hidden, cell) = self.lstm(
            X
        )


        if self.bidirectional:

            hidden = torch.cat(
                (
                    hidden[-2],
                    hidden[-1]
                ),
                dim=1
            )

        else:

            hidden = hidden[-1]


        hidden = self.dropout(
            hidden
        )


        X = self.fc(
            hidden
        )


        return X

In [ ]:
class GRUModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=100,
        hidden_dim=128,
        output_dim=3,
        dropout=0.3,
        bidirectional=False,
        padding_idx=0
    ):

        super().__init__()

        self.bidirectional = bidirectional


        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=padding_idx
        )


        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=bidirectional
        )


        self.dropout = nn.Dropout(
            dropout
        )


        direction = (
            2 if bidirectional
            else 1
        )


        self.fc = nn.Linear(
            hidden_dim * direction,
            output_dim
        )


    def forward(self, X):

        X = self.embedding(X)


        output, hidden = self.gru(
            X
        )


        if self.bidirectional:

            hidden = torch.cat(
                (
                    hidden[-2],
                    hidden[-1]
                ),
                dim=1
            )

        else:

            hidden = hidden[-1]


        hidden = self.dropout(
            hidden
        )


        X = self.fc(
            hidden
        )


        return X

# Parameter


In [ ]:
param_grid = {
    "embedding_dim": [128, 160],
    "hidden_dim": [64, 128, 192],
    "dropout": [0.2, 0.3, 0.4],
    "learning_rate": [0.001, 0.0007],
    "bidirectional": [True]
}

In [ ]:
from itertools import product


def generate_combinations(param_grid):

    keys = param_grid.keys()
    values = param_grid.values()

    combinations = []

    for combination in product(*values):

        params = dict(
            zip(keys, combination)
        )

        combinations.append(params)

    return combinations

In [ ]:
parameter_combinations = generate_combinations(
    param_grid
)

print(
    "Total parameter combinations:",
    len(parameter_combinations)
)

Total parameter combinations: 36


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy


def grid_search_model(
    model_class,
    model_name,
    param_grid,
    vocab_size,
    train_loader,
    val_loader,
    class_weights,
    num_epochs=100,
    patience=5
):

    # =====================================================
    # 1. GENERATE ALL PARAMETER COMBINATIONS
    # =====================================================

    combinations = generate_combinations(
        param_grid
    )


    # =====================================================
    # 2. VARIABLES TO STORE BEST RESULT
    # =====================================================

    # Higher Macro F1 is better
    best_val_f1 = -float("inf")

    best_params = None

    best_model_state = None


    print(
        f"\n========== {model_name} GRID SEARCH =========="
    )


    # =====================================================
    # 3. LOOP THROUGH PARAMETER COMBINATIONS
    # =====================================================

    for i, params in enumerate(
        combinations,
        start=1
    ):

        print(
            f"\nCombination "
            f"{i}/{len(combinations)}"
        )

        print(
            params
        )


        # =================================================
        # 4. CREATE FRESH MODEL
        # =================================================

        model = model_class(
            vocab_size=vocab_size,

            embedding_dim=params[
                "embedding_dim"
            ],

            hidden_dim=params[
                "hidden_dim"
            ],

            output_dim=3,

            dropout=params[
                "dropout"
            ],

            bidirectional=params[
                "bidirectional"
            ],

            padding_idx=0
        )


        model = model.to(
            device
        )


        # =================================================
        # 5. WEIGHTED CROSS ENTROPY LOSS
        # =================================================

        criterion = nn.CrossEntropyLoss(
            weight=class_weights.to(
                device
            )
        )


        # =================================================
        # 6. OPTIMIZER
        # =================================================

        optimizer = optim.Adam(
            model.parameters(),

            lr=params[
                "learning_rate"
            ]
        )


        # =================================================
        # 7. TRAIN MODEL
        # =================================================
        #
        # train_model() uses Validation Macro F1
        # for early stopping.
        #
        # It restores the weights from the epoch
        # with the best Validation Macro F1.
        # =================================================

        model = train_model(
            model=model,

            train_loader=train_loader,

            val_loader=val_loader,

            criterion=criterion,

            optimizer=optimizer,

            num_epochs=num_epochs,

            patience=patience
        )


        # =================================================
        # 8. EVALUATE BEST EPOCH ON VALIDATION SET
        # =================================================

        metrics = evaluate_model(
            model,
            val_loader
        )


        # evaluate_model() returns Macro F1
        current_val_f1 = metrics[
            "F1"
        ]


        print(
            f"\nValidation Macro F1: "
            f"{current_val_f1:.4f}"
        )


        # =================================================
        # 9. COMPARE WITH BEST PARAMETER COMBINATION
        # =================================================

        if current_val_f1 > best_val_f1:

            # Save best validation Macro F1
            best_val_f1 = current_val_f1


            # Save best hyperparameters
            best_params = params.copy()


            # Save best model weights
            best_model_state = copy.deepcopy(
                model.state_dict()
            )


            print(
                "New best parameter combination found!"
            )


    # =====================================================
    # 10. DISPLAY GRID SEARCH RESULT
    # =====================================================

    print(
        f"\n========== BEST {model_name} =========="
    )


    print(
        "Best Parameters:",
        best_params
    )


    print(
        f"Best Validation Macro F1 Score: "
        f"{best_val_f1:.4f}"
    )


    # =====================================================
    # 11. RETURN RESULTS
    # =====================================================

    return (
        best_params,
        best_val_f1,
        best_model_state
    )

# RNN

In [ ]:
rnn_best_params, rnn_best_f1, rnn_best_state = (
    grid_search_model(
        model_class=RNNModel,
        model_name="RNN",
        param_grid=param_grid,
        vocab_size=vocab_size,
        train_loader=train_loader,
        val_loader=val_loader,
        class_weights=class_weights,
        num_epochs=100,
        patience=5
    )
)


========== RNN GRID SEARCH ==========

Combination 1/36
{'embedding_dim': 128, 'hidden_dim': 64, 'dropout': 0.2, 'learning_rate': 0.001, 'bidirectional': True}
Epoch 1/100 --> Train Loss: 1.0867 --> Train Accuracy: 0.4150 --> Val Loss: 1.0527 --> Val Accuracy: 0.5128 --> Val Macro F1: 0.4225
Validation Macro F1 improved to 0.4225
Epoch 2/100 --> Train Loss: 0.9695 --> Train Accuracy: 0.5737 --> Val Loss: 0.9901 --> Val Accuracy: 0.5919 --> Val Macro F1: 0.4959
Validation Macro F1 improved to 0.4959
Epoch 3/100 --> Train Loss: 0.8051 --> Train Accuracy: 0.6801 --> Val Loss: 0.9774 --> Val Accuracy: 0.5966 --> Val Macro F1: 0.5178
Validation Macro F1 improved to 0.5178
Epoch 4/100 --> Train Loss: 0.6291 --> Train Accuracy: 0.7642 --> Val Loss: 0.9947 --> Val Accuracy: 0.6113 --> Val Macro F1: 0.5309
Validation Macro F1 improved to 0.5309
Epoch 5/100 --> Train Loss: 0.4660 --> Train Accuracy: 0.8322 --> Val Loss: 1.0624 --> Val Accuracy: 0.6218 --> Val Macro F1: 0.5466
Validation Macro F

In [ ]:
import pickle

save_path = "/content/drive/MyDrive/Financial_News_Sentiment_Prediction_using_DeepLearning/best_rnn_params.pkl"

with open(save_path, "wb") as f:
    pickle.dump(rnn_best_params, f)

print("Saved at:", save_path)

Saved at: /content/drive/MyDrive/Financial_News_Sentiment_Prediction_using_DeepLearning/best_rnn_params.pkl


# LSTM

In [ ]:
lstm_best_params, lstm_best_f1, lstm_best_state = (
    grid_search_model(
        model_class=LSTMModel,
        model_name="LSTM",
        param_grid=param_grid,
        vocab_size=vocab_size,
        train_loader=train_loader,
        val_loader=val_loader,
        class_weights=class_weights,
        num_epochs=100,
        patience=5
    )
)


========== LSTM GRID SEARCH ==========

Combination 1/36
{'embedding_dim': 128, 'hidden_dim': 64, 'dropout': 0.2, 'learning_rate': 0.001, 'bidirectional': True}
Epoch 1/100 --> Train Loss: 1.0638 --> Train Accuracy: 0.4893 --> Val Loss: 0.9917 --> Val Accuracy: 0.5825 --> Val Macro F1: 0.4740
Validation Macro F1 improved to 0.4740
Epoch 2/100 --> Train Loss: 0.8642 --> Train Accuracy: 0.6948 --> Val Loss: 0.8839 --> Val Accuracy: 0.6789 --> Val Macro F1: 0.5706
Validation Macro F1 improved to 0.5706
Epoch 3/100 --> Train Loss: 0.6199 --> Train Accuracy: 0.7891 --> Val Loss: 0.8517 --> Val Accuracy: 0.7056 --> Val Macro F1: 0.6118
Validation Macro F1 improved to 0.6118
Epoch 4/100 --> Train Loss: 0.3990 --> Train Accuracy: 0.8740 --> Val Loss: 0.8855 --> Val Accuracy: 0.7129 --> Val Macro F1: 0.6224
Validation Macro F1 improved to 0.6224
Epoch 5/100 --> Train Loss: 0.2352 --> Train Accuracy: 0.9264 --> Val Loss: 0.9927 --> Val Accuracy: 0.7171 --> Val Macro F1: 0.6391
Validation Macro 

In [ ]:
import pickle

save_path = "/content/drive/MyDrive/Financial_News_Sentiment_Prediction_using_DeepLearning/best_lstm_params.pkl"

with open(save_path, "wb") as f:
    pickle.dump(lstm_best_params, f)

print("Saved at:", save_path)

Saved at: /content/drive/MyDrive/Financial_News_Sentiment_Prediction_using_DeepLearning/best_lstm_params.pkl


# GRU

In [ ]:
gru_best_params, gru_best_f1, gru_best_state = grid_search_model(
    model_class=GRUModel,
    model_name="GRU",
    param_grid=param_grid,
    vocab_size=vocab_size,
    train_loader=train_loader,
    val_loader=val_loader,
    class_weights=class_weights,
    num_epochs=100,
    patience=5
)


========== GRU GRID SEARCH ==========

Combination 1/36
{'embedding_dim': 128, 'hidden_dim': 64, 'dropout': 0.2, 'learning_rate': 0.001, 'bidirectional': True}
Epoch 1/100 --> Train Loss: 1.0653 --> Train Accuracy: 0.4564 --> Val Loss: 0.9990 --> Val Accuracy: 0.5605 --> Val Macro F1: 0.4650
Validation Macro F1 improved to 0.4650
Epoch 2/100 --> Train Loss: 0.8705 --> Train Accuracy: 0.6589 --> Val Loss: 0.8844 --> Val Accuracy: 0.5982 --> Val Macro F1: 0.5407
Validation Macro F1 improved to 0.5407
Epoch 3/100 --> Train Loss: 0.6252 --> Train Accuracy: 0.7664 --> Val Loss: 0.8244 --> Val Accuracy: 0.7056 --> Val Macro F1: 0.6259
Validation Macro F1 improved to 0.6259
Epoch 4/100 --> Train Loss: 0.3975 --> Train Accuracy: 0.8600 --> Val Loss: 0.9269 --> Val Accuracy: 0.7192 --> Val Macro F1: 0.6360
Validation Macro F1 improved to 0.6360
Epoch 5/100 --> Train Loss: 0.2341 --> Train Accuracy: 0.9259 --> Val Loss: 1.0057 --> Val Accuracy: 0.7297 --> Val Macro F1: 0.6453
Validation Macro F

In [ ]:
import pickle

save_path = "/content/drive/MyDrive/Financial_News_Sentiment_Prediction_using_DeepLearning/best_gru_params.pkl"

with open(save_path, "wb") as f:
    pickle.dump(gru_best_params, f)

print("Saved at:", save_path)

Saved at: /content/drive/MyDrive/Financial_News_Sentiment_Prediction_using_DeepLearning/best_gru_params.pkl


# Read the parameters

In [ ]:
import pickle

base_path = "/content/drive/MyDrive/Financial_News_Sentiment_Prediction_using_DeepLearning"


# Load RNN parameters
with open(
    f"{base_path}/best_rnn_params.pkl",
    "rb"
) as f:
    best_rnn_params = pickle.load(f)


# Load LSTM parameters
with open(
    f"{base_path}/best_lstm_params.pkl",
    "rb"
) as f:
    best_lstm_params = pickle.load(f)


# Load GRU parameters
with open(
    f"{base_path}/best_gru_params.pkl",
    "rb"
) as f:
    best_gru_params = pickle.load(f)

In [ ]:
print("Best RNN Parameters:")
print(best_rnn_params)

print("\nBest LSTM Parameters:")
print(best_lstm_params)

print("\nBest GRU Parameters:")
print(best_gru_params)

Best RNN Parameters:
{'embedding_dim': 160, 'hidden_dim': 64, 'dropout': 0.2, 'learning_rate': 0.001, 'bidirectional': True}

Best LSTM Parameters:
{'embedding_dim': 160, 'hidden_dim': 128, 'dropout': 0.4, 'learning_rate': 0.001, 'bidirectional': True}

Best GRU Parameters:
{'embedding_dim': 160, 'hidden_dim': 128, 'dropout': 0.3, 'learning_rate': 0.001, 'bidirectional': True}


# model creation

In [ ]:
# =========================
# FINAL RNN
# =========================

rnn_model = RNNModel(
    vocab_size=vocab_size,
    embedding_dim=best_rnn_params["embedding_dim"],
    hidden_dim=best_rnn_params["hidden_dim"],
    output_dim=3,
    dropout=best_rnn_params["dropout"],
    bidirectional=best_rnn_params["bidirectional"]
).to(device)

rnn_optimizer = optim.Adam(
    rnn_model.parameters(),
    lr=best_rnn_params["learning_rate"]
)


# =========================
# FINAL LSTM
# =========================

lstm_model = LSTMModel(
    vocab_size=vocab_size,
    embedding_dim=best_lstm_params["embedding_dim"],
    hidden_dim=best_lstm_params["hidden_dim"],
    output_dim=3,
    dropout=best_lstm_params["dropout"],
    bidirectional=best_lstm_params["bidirectional"]
).to(device)

lstm_optimizer = optim.Adam(
    lstm_model.parameters(),
    lr=best_lstm_params["learning_rate"]
)


# =========================
# FINAL GRU
# =========================

gru_model = GRUModel(
    vocab_size=vocab_size,
    embedding_dim=best_gru_params["embedding_dim"],
    hidden_dim=best_gru_params["hidden_dim"],
    output_dim=3,
    dropout=best_gru_params["dropout"],
    bidirectional=best_gru_params["bidirectional"]
).to(device)

gru_optimizer = optim.Adam(
    gru_model.parameters(),
    lr=best_gru_params["learning_rate"]
)

In [ ]:
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

# RNN

In [ ]:
print("\n========== FINAL RNN TRAINING ==========")

rnn_history = train_model(
    model=rnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=rnn_optimizer,
    num_epochs=50,
    patience=5
)


========== FINAL RNN TRAINING ==========
Epoch 1/50 --> Train Loss: 1.0875 --> Train Accuracy: 0.4091 --> Val Loss: 1.0513 --> Val Accuracy: 0.5411 --> Val Macro F1: 0.4245
Validation Macro F1 improved to 0.4245
Epoch 2/50 --> Train Loss: 0.9780 --> Train Accuracy: 0.5817 --> Val Loss: 0.9753 --> Val Accuracy: 0.5825 --> Val Macro F1: 0.4885
Validation Macro F1 improved to 0.4885
Epoch 3/50 --> Train Loss: 0.8067 --> Train Accuracy: 0.6931 --> Val Loss: 0.9521 --> Val Accuracy: 0.6757 --> Val Macro F1: 0.5616
Validation Macro F1 improved to 0.5616
Epoch 4/50 --> Train Loss: 0.6028 --> Train Accuracy: 0.7949 --> Val Loss: 0.9335 --> Val Accuracy: 0.6459 --> Val Macro F1: 0.5668
Validation Macro F1 improved to 0.5668
Epoch 5/50 --> Train Loss: 0.4151 --> Train Accuracy: 0.8652 --> Val Loss: 1.0172 --> Val Accuracy: 0.6778 --> Val Macro F1: 0.5951
Validation Macro F1 improved to 0.5951
Epoch 6/50 --> Train Loss: 0.2680 --> Train Accuracy: 0.9164 --> Val Loss: 1.1704 --> Val Accuracy: 0.7

LSTM

In [ ]:
print("\n========== FINAL LSTM TRAINING ==========")

lstm_history = train_model(
    model=lstm_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=lstm_optimizer,
    num_epochs=50,
    patience=5
)


========== FINAL LSTM TRAINING ==========
Epoch 1/50 --> Train Loss: 1.0319 --> Train Accuracy: 0.5308 --> Val Loss: 0.9298 --> Val Accuracy: 0.6338 --> Val Macro F1: 0.5284
Validation Macro F1 improved to 0.5284
Epoch 2/50 --> Train Loss: 0.7761 --> Train Accuracy: 0.7173 --> Val Loss: 0.8367 --> Val Accuracy: 0.7067 --> Val Macro F1: 0.6089
Validation Macro F1 improved to 0.6089
Epoch 3/50 --> Train Loss: 0.5202 --> Train Accuracy: 0.8184 --> Val Loss: 0.8529 --> Val Accuracy: 0.7239 --> Val Macro F1: 0.6274
Validation Macro F1 improved to 0.6274
Epoch 4/50 --> Train Loss: 0.3022 --> Train Accuracy: 0.9002 --> Val Loss: 0.9416 --> Val Accuracy: 0.7124 --> Val Macro F1: 0.6302
Validation Macro F1 improved to 0.6302
Epoch 5/50 --> Train Loss: 0.1554 --> Train Accuracy: 0.9506 --> Val Loss: 1.2587 --> Val Accuracy: 0.7522 --> Val Macro F1: 0.6590
Validation Macro F1 improved to 0.6590
Epoch 6/50 --> Train Loss: 0.0747 --> Train Accuracy: 0.9776 --> Val Loss: 1.3045 --> Val Accuracy: 0.

GRU

In [ ]:
print("\n========== FINAL GRU TRAINING ==========")

gru_history = train_model(
    model=gru_model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=gru_optimizer,
    num_epochs=50,
    patience=5
)


========== FINAL GRU TRAINING ==========
Epoch 1/50 --> Train Loss: 1.0221 --> Train Accuracy: 0.5140 --> Val Loss: 0.9191 --> Val Accuracy: 0.6438 --> Val Macro F1: 0.5419
Validation Macro F1 improved to 0.5419
Epoch 2/50 --> Train Loss: 0.7526 --> Train Accuracy: 0.7081 --> Val Loss: 0.8211 --> Val Accuracy: 0.6852 --> Val Macro F1: 0.6019
Validation Macro F1 improved to 0.6019
Epoch 3/50 --> Train Loss: 0.4898 --> Train Accuracy: 0.8233 --> Val Loss: 0.8287 --> Val Accuracy: 0.6920 --> Val Macro F1: 0.6258
Validation Macro F1 improved to 0.6258
Epoch 4/50 --> Train Loss: 0.2882 --> Train Accuracy: 0.8977 --> Val Loss: 0.9382 --> Val Accuracy: 0.7213 --> Val Macro F1: 0.6447
Validation Macro F1 improved to 0.6447
Epoch 5/50 --> Train Loss: 0.1435 --> Train Accuracy: 0.9515 --> Val Loss: 1.1382 --> Val Accuracy: 0.7417 --> Val Macro F1: 0.6547
Validation Macro F1 improved to 0.6547
Epoch 6/50 --> Train Loss: 0.0626 --> Train Accuracy: 0.9805 --> Val Loss: 1.3606 --> Val Accuracy: 0.7

# Evaluation


In [ ]:
print("\n========== RNN TEST EVALUATION ==========")

rnn_results = evaluate_model(
    model=rnn_model,
    test_loader=test_loader,
    device=device
)


print("\n========== LSTM TEST EVALUATION ==========")

lstm_results = evaluate_model(
    model=lstm_model,
    test_loader=test_loader,
    device=device
)


print("\n========== GRU TEST EVALUATION ==========")

gru_results = evaluate_model(
    model=gru_model,
    test_loader=test_loader,
    device=device
)


========== RNN TEST EVALUATION ==========
Accuracy  : 0.7546
Precision : 0.6647
Recall    : 0.6116
F1 Score  : 0.6325

Classification Report:
              precision    recall  f1-score   support

     Bearish       0.55      0.44      0.49       347
     Bullish       0.63      0.49      0.55       475
     Neutral       0.81      0.90      0.86      1566

    accuracy                           0.75      2388
   macro avg       0.66      0.61      0.63      2388
weighted avg       0.74      0.75      0.74      2388

Confusion Matrix:
[[ 152   56  139]
 [  56  234  185]
 [  66   84 1416]]

========== LSTM TEST EVALUATION ==========
Accuracy  : 0.7688
Precision : 0.7065
Recall    : 0.6104
F1 Score  : 0.6432

Classification Report:
              precision    recall  f1-score   support

     Bearish       0.60      0.41      0.49       347
     Bullish       0.72      0.48      0.58       475
     Neutral       0.80      0.93      0.86      1566

    accuracy                           0.

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame({
    "RNN": rnn_results,
    "LSTM": lstm_results,
    "GRU": gru_results
}).T

comparison_df = comparison_df.round(4)

print("\n========================================")
print("MODEL COMPARISON")
print("========================================")

print(comparison_df)


MODEL COMPARISON
      accuracy precision    recall        f1  \
RNN   0.754606  0.664735  0.611629  0.632465   
LSTM  0.768844  0.706454  0.610439  0.643227   
GRU   0.724456  0.647033  0.655174  0.639153   

                                      confusion_matrix  \
RNN   [[152, 56, 139], [56, 234, 185], [66, 84, 1416]]   
LSTM  [[144, 37, 166], [45, 229, 201], [52, 51, 1463]]   
GRU   [[217, 42, 88], [95, 255, 125], [235, 73, 1258]]   

                                                 actual  \
RNN   [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   
LSTM  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   
GRU   [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   

                                            predictions  
RNN   [1, 2, 0, 0, 2, 0, 2, 2, 1, 2, 0, 0, 2, 2, 0, ...  
LSTM  [2, 0, 0, 0, 1, 0, 2, 2, 2, 2, 0, 0, 2, 0, 0, ...  
GRU   [2, 2, 0, 0, 0, 0, 0, 2, 0, 2, 0, 0, 0, 1, 0, ...  


In [ ]:
import os
import torch
import joblib

# Google Drive project folder
model_path = "/content/drive/MyDrive/Financial_News_Sentiment_Prediction_using_DeepLearning"

# Create folder if it doesn't exist
os.makedirs(model_path, exist_ok=True)

# 1. Save trained LSTM model
torch.save(
    lstm_model.state_dict(),
    os.path.join(model_path, "lstm_model.pth")
)

# 2. Save vocabulary
joblib.dump(
    word_to_index,
    os.path.join(model_path, "word_to_index.pkl")
)

# 3. Save label mapping
label_mapping = {
    0: "Bearish",
    1: "Bullish",
    2: "Neutral"
}

joblib.dump(
    label_mapping,
    os.path.join(model_path, "label_mapping.pkl")
)

# 4. Save LSTM configuration using BEST parameters
lstm_config = {
    "vocab_size": vocab_size,
    "embedding_dim": best_lstm_params["embedding_dim"],
    "hidden_dim": best_lstm_params["hidden_dim"],
    "output_dim": 3,
    "dropout": best_lstm_params["dropout"],
    "bidirectional": best_lstm_params["bidirectional"],
    "padding_idx": 0,
    "max_length": max_length
}

joblib.dump(
    lstm_config,
    os.path.join(model_path, "lstm_config.pkl")
)

print("All LSTM files saved successfully!")

print("\nBest LSTM Parameters:")
print(best_lstm_params)

print("\nSaved location:")
print(model_path)

print("\nFiles:")
print(os.listdir(model_path))

All LSTM files saved successfully!

Best LSTM Parameters:
{'embedding_dim': 160, 'hidden_dim': 128, 'dropout': 0.4, 'learning_rate': 0.001, 'bidirectional': True}

Saved location:
/content/drive/MyDrive/Financial_News_Sentiment_Prediction_using_DeepLearning

Files:
['README.md', 'cleaned_test.csv', 'best_rnn_params.pkl', 'project_summary.docx', 'label_mapping.pkl', 'lstm_config.pkl', 'word_to_index.pkl', 'best_gru_params.pkl', 'sent_valid.csv', 'Data_cleaning.ipynb', 'best_lstm_params.pkl', 'lstm_model.pth', 'streamlit.ipynb', 'EDA.ipynb', 'requirements.txt', 'business_case_data.csv', 'rnn,lstm,gru.ipynb', 'sent_train.csv', 'cleaned.csv', '.git', 'models']
